<a href="https://colab.research.google.com/github/rafayyk7/flyrank-ml/blob/main/work/notebooks/w02_prompt_ladder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🪜 Week 2 Assignment: The Prompt Ladder
**Track:** Machine Learning Foundations & Capstone Pipeline  

---

## 🛑 Run 0: The Baseline (The Weak Prompt)
> **Prompt:** `"Write python code to predict content decay."`

```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv('data.csv')
X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_score=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)
preds = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, preds))

# **🧱 Run 1: Adding a Defined Persona**
Added Layer: Defined Audience / Persona Constraint

Prompt: "Act as a Senior Machine Learning Engineer reviewing a capstone project. Write python code to predict content decay."

📝 Notes:
What Changed in Prompt: Added persona ("Act as a Senior Machine Learning Engineer...").

What Improved in Output: Wrapped logic inside a type-hinted function and upgraded metric to classification_report.

What Still Failed: Data inputs were generic assumptions ('clicks', 'impressions').

What to Try Next: Add real domain context (Google Search Console data).

# **🧱 Run 2: Adding Real Domain Context**
Added Layer: Real Context

Prompt: "Act as a Senior Machine Learning Engineer reviewing a capstone project. We are working with Google Search Console data across 32 domains. Write python code to predict whether a URL's rolling 30-day organic impressions will experience significant decay relative to its historical baseline."

📝 Notes:
What Changed in Prompt: Added real domain context ("Google Search Console data across 32 domains").

What Improved in Output: Engineered realistic SEO features (impression_momentum, position_volatility).

What Still Failed: Returned fragmented snippets without a clear DataFrame matrix structure.

What to Try Next: Specify an output format requiring a structured pandas DataFrame.

# **🧱 Run 3: Adding a Specified Output Format**
Added Layer: Specified Output Format

Prompt: "Act as a Senior Machine Learning Engineer reviewing a capstone project. We are working with Google Search Console data across 32 domains. Write python code to predict whether a URL's rolling 30-day organic impressions will experience significant decay relative to its historical baseline. Format the output as a standalone Python script that builds a mock DataFrame representing the exact Unit of Analysis (One row = One unique URL per snapshot date) and prints its schema."

📝 Notes:
What Changed in Prompt: Specified explicit output format (standalone script showing Unit of Analysis schema).

What Improved in Output: Generated a structured pandas table explicitly showing primary keys and .info() schema.

What Still Failed: Used leaky train_test_split which bleeds domain authority signals across splits.

What to Try Next: Add strict validation constraints preventing target leakage.

# **🧱 Run 4: Adding Strict Validation Constraints**
Added Layer: Specific Constraints

Prompt: "Act as a Senior Machine Learning Engineer reviewing a capstone project. We are working with Google Search Console data across 32 domains. Write python code to predict whether a URL's rolling 30-day organic impressions will experience significant decay relative to its historical baseline. Format the output as a standalone Python script that builds a mock DataFrame representing the exact Unit of Analysis (One row = One unique URL per snapshot date) and prints its schema. Constraint: You must use GroupKFold cross-validation grouped strictly by 'domain_id' to prevent target leakage across website domains. Do NOT use standard train_test_split."

📝 Notes (Honest Failure Moment):
What Changed in Prompt: Added constraint (GroupKFold grouped strictly by domain_id).

What Actually Improved in Output: Replaced leaky split logic with domain-grouped evaluation.

What Failed / Made It Worse: The code crashed with a ValueError. The model created a 3-fold split loop that failed on its own 2-domain mock dataset because n_splits exceeded the number of groups.

What to Try Next: Add quality criteria and align evaluation metrics with business rewrite capacity.

# **🧱 Run 5: Adding Quality Criteria & Business Metric Alignment**
Added Layer: Quality Criteria & Metric Alignment

Prompt: "Act as a Senior Machine Learning Engineer reviewing a capstone project. We are working with Google Search Console data across 32 domains. Write python code to predict whether a URL's rolling 30-day organic impressions will experience significant decay relative to its historical baseline. Format the output as a standalone Python script that builds a mock DataFrame representing the exact Unit of Analysis (One row = One unique URL per snapshot date) and prints its schema. Constraint: You must use GroupKFold cross-validation grouped strictly by 'domain_id' to prevent target leakage across website domains. Do NOT use standard train_test_split. Quality Criterion: Evaluate the model using Precision@50 to reflect a strict business constraint where a content operations team has a budget of only 50 monthly page rewrites."

📝 Notes:
What Changed in Prompt: Added Precision@50 metric linked to the 50-rewrite monthly budget constraint.

What Improved in Output: Added custom precision_at_k metric and expanded mock data to 60 rows across 5 domains so code executes cleanly.

What Still Failed: Required context known only to the user; needed standard prompt template formatting.

What to Try Next: Standardize into a reusable production prompt template.

Role & Persona:
Act as a Principal Machine Learning Engineer specializing in SEO data pipelines and tabular time-series modeling.

Context & Problem Statement:
We are building a Content Decay prediction engine using Google Search Console performance data across multiple client domains. A web page is flagged as "decayed" if its rolling 30-day organic impressions drop significantly relative to its historical baseline.

Task:
Write a clean, modular, and fully executable Python script that frames this problem, builds a synthetic dataset matching our exact Unit of Analysis, and evaluates a baseline classifier without data leakage.

Required Technical Specifications:
1. Unit of Analysis: One row = One unique URL per historical snapshot date. Include columns for `snapshot_date`, `domain_id`, `url`, feature columns (`impressions_90d`, `avg_position`, `position_volatility`, `impression_momentum_ratio`), and target label `is_decayed` (binary 0 or 1).
2. Validation Strategy (NO TARGET LEAKAGE): You MUST use GroupKFold cross-validation grouped strictly by `domain_id`. Do NOT use standard `train_test_split`, as sitewide domain signals will leak into the test set.
3. Optimization Metric: Implement a custom Precision@K metric function (Precision@50). Explain in a code comment why Precision@K is the correct metric for content operations teams constrained by fixed monthly rewrite budgets.
4. Execution: Ensure the script generates sufficient mock data across at least 5 unique domains so the script runs cleanly end-to-end without index or split errors.

Output Requirements:
Return a single, well-commented Python file with type hints, structured functions, and clear print statements showing the dataset schema and holdout validation scores.

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold

# 1. Custom Precision@K Metric
def precision_at_k(y_true, y_probs, k=10):
    """Calculates Precision@K matching fixed operational rewrite capacity."""
    top_k_indices = np.argsort(y_probs)[::-1][:k]
    return np.mean(y_true.iloc[top_k_indices])

# 2. Synthetic Dataset matching Unit of Analysis
np.random.seed(42)
domains = [f'domain_{i:02d}' for i in range(1, 6)]
df = pd.DataFrame({
    'snapshot_date': '2026-07-15',
    'domain_id': np.random.choice(domains, 60),
    'url': [f'https://example.com/page_{i}' for i in range(60)],
    'impressions_90d': np.random.randint(1000, 50000, 60),
    'avg_position': np.random.uniform(1.0, 30.0, 60),
    'position_volatility': np.random.uniform(0.1, 5.0, 60),
    'impression_momentum_ratio': np.random.uniform(0.3, 1.2, 60),
    'is_decayed': np.random.choice([0, 1], 60, p=[0.6, 0.4])
})

print("--- UNIT OF ANALYSIS SCHEMA ---")
print(df.info())

# 3. Leakage-Free GroupKFold Evaluation
gkf = GroupKFold(n_splits=3)
X = df[['impressions_90d', 'avg_position', 'position_volatility', 'impression_momentum_ratio']]
y = df['is_decayed']
groups = df['domain_id']

scores = []
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups), 1):
    clf = RandomForestClassifier(n_estimators=50, random_state=42)
    clf.fit(X.iloc[train_idx], y.iloc[train_idx])
    probs = clf.predict_proba(X.iloc[val_idx])[:, 1]

    p_at_k = precision_at_k(y.iloc[val_idx], probs, k=5)
    scores.append(p_at_k)
    print(f"Fold {fold} Holdout Precision@K: {p_at_k:.2f}")

print(f"\nMean Holdout Precision@K across unseen domains: {np.mean(scores):.2f}")

--- UNIT OF ANALYSIS SCHEMA ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   snapshot_date              60 non-null     object 
 1   domain_id                  60 non-null     object 
 2   url                        60 non-null     object 
 3   impressions_90d            60 non-null     int64  
 4   avg_position               60 non-null     float64
 5   position_volatility        60 non-null     float64
 6   impression_momentum_ratio  60 non-null     float64
 7   is_decayed                 60 non-null     int64  
dtypes: float64(3), int64(2), object(3)
memory usage: 3.9+ KB
None
Fold 1 Holdout Precision@K: 0.20
Fold 2 Holdout Precision@K: 0.60
Fold 3 Holdout Precision@K: 0.60

Mean Holdout Precision@K across unseen domains: 0.47
